This notebook requires the ``sampled_data.csv`` from ``02_sample_data.py``.

In [46]:
import pandas as pd

df = pd.read_csv('data/sampled_data.csv')
weather = pd.read_csv('data/weather.csv')
df.head()

C:\Users\l\AppData\Local\Temp\ipykernel_73300\1160607818.py:3: DtypeWarning: Columns (0: start_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/sampled_data.csv')


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,4C1563485B2BC1B5,classic_bike,2021-01-01 13:02:20.043,2021-01-01 13:06:54.252,Willis Ave & E 141 St,7759.08,Morris Ave & E 142 St,7821.01,40.810930,-73.921606,40.814469,-73.924963,member
1,6FB71DDD88A98375,electric_bike,2021-01-01 12:51:05.140,2021-01-01 13:08:58.948,S 4 St & Wythe Ave,5204.05,Grand Army Plaza & Plaza St West,4010.15,40.712859,-73.965903,40.672968,-73.970880,member
2,37C923C6A52E6886,electric_bike,2021-01-26 20:39:37.656,2021-01-26 20:42:27.453,E 51 St & 1 Ave,6532.06,E 56 St & 3 Ave,6691.11,40.754557,-73.965930,40.759345,-73.967597,member
3,5E244CCFE0FDD8B4,classic_bike,2021-01-11 12:31:31.977,2021-01-11 12:37:50.308,10 Ave & W 28 St,6459.04,W 18 St & 9 Ave,6190.03,40.750664,-74.001768,40.743534,-74.003676,member
4,F90710E50A175D34,classic_bike,2021-01-08 12:12:48.360,2021-01-08 12:22:27.454,Bond St & Fulton St,4479.06,Front St & Washington St,4936.01,40.689622,-73.983043,40.702551,-73.989402,member


In [47]:
# 2. Clean the data

df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
df['ended_at'] = pd.to_datetime(df['ended_at'], errors='coerce')
df['trip_duration_min'] = (df['ended_at'] - df['started_at']).dt.total_seconds() / 60

dt = df['started_at'].dt
df['hour'] = dt.hour
df['day_of_week'] = dt.day_name()
df['month'] = dt.month_name()

df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday'])

month_map = {
    "December": "Winter", "January": "Winter", "February": "Winter",
    "March": "Spring", "April": "Spring", "May": "Spring",
    "June": "Summer", "July": "Summer", "August": "Summer",
    "September": "Fall", "October": "Fall", "November": "Fall"
}
df['season'] = df['month'].map(month_map)

df = df[(df['trip_duration_min'] > 0) & (df['trip_duration_min'] < 60)]


In [48]:
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,trip_duration_min,hour,day_of_week,month,is_weekend,season
0,4C1563485B2BC1B5,classic_bike,2021-01-01 13:02:20.043,2021-01-01 13:06:54.252,Willis Ave & E 141 St,7759.08,Morris Ave & E 142 St,7821.01,40.810930,-73.921606,40.814469,-73.924963,member,4.570150,13,Friday,January,False,Winter
1,6FB71DDD88A98375,electric_bike,2021-01-01 12:51:05.140,2021-01-01 13:08:58.948,S 4 St & Wythe Ave,5204.05,Grand Army Plaza & Plaza St West,4010.15,40.712859,-73.965903,40.672968,-73.970880,member,17.896800,12,Friday,January,False,Winter
2,37C923C6A52E6886,electric_bike,2021-01-26 20:39:37.656,2021-01-26 20:42:27.453,E 51 St & 1 Ave,6532.06,E 56 St & 3 Ave,6691.11,40.754557,-73.965930,40.759345,-73.967597,member,2.829950,20,Tuesday,January,False,Winter
3,5E244CCFE0FDD8B4,classic_bike,2021-01-11 12:31:31.977,2021-01-11 12:37:50.308,10 Ave & W 28 St,6459.04,W 18 St & 9 Ave,6190.03,40.750664,-74.001768,40.743534,-74.003676,member,6.305517,12,Monday,January,False,Winter
4,F90710E50A175D34,classic_bike,2021-01-08 12:12:48.360,2021-01-08 12:22:27.454,Bond St & Fulton St,4479.06,Front St & Washington St,4936.01,40.689622,-73.983043,40.702551,-73.989402,member,9.651567,12,Friday,January,False,Winter


In [ ]:
(df.isnull().sum() / len(df)) * 100 


ride_id               0.000000
rideable_type         0.000000
started_at            0.000000
ended_at              0.000000
start_station_name    0.000752
start_station_id      0.000752
end_station_name      0.186594
end_station_id        0.186594
start_lat             0.000000
start_lng             0.000000
end_lat               0.115116
end_lng               0.115116
member_casual         0.000000
trip_duration_min     0.000000
hour                  0.000000
day_of_week           0.000000
month                 0.000000
is_weekend            0.000000
season                0.000000
dtype: float64

In [50]:
def fill_station_info(df, prefix='end'):

    # Build lookup table for the station
    station_lookup = df.dropna(subset=[f'{prefix}_station_name'])[
        [f'{prefix}_station_name', f'{prefix}_station_id', f'{prefix}_lat', f'{prefix}_lng']
    ].drop_duplicates()
    
    # Merge with original dataframe
    df = df.merge(
        station_lookup,
        on=[f'{prefix}_lat', f'{prefix}_lng'],
        how='left',
        suffixes=('', '_filled')
    )
    
    # Fill missing values
    df[f'{prefix}_station_name'] = df[f'{prefix}_station_name'].fillna(df[f'{prefix}_station_name_filled'])
    df[f'{prefix}_station_id'] = df[f'{prefix}_station_id'].fillna(df[f'{prefix}_station_id_filled'])
    
    # Drop temporary columns
    df = df.drop(columns=[f'{prefix}_station_name_filled', f'{prefix}_station_id_filled'])
    
    return df

df = fill_station_info(df, 'start')
df = fill_station_info(df, 'end')

df = df.dropna()

In [51]:
# 2.1 Ckean weather data
weather['time'] = pd.to_datetime(weather['time'])
weather['date'] = weather['time'].dt.date
weather_daily = weather.groupby('date').agg({
    'temperature_2m (°C)': 'mean',
    'precipitation (mm)': 'sum',
    'rain (mm)': 'sum',
    'cloudcover (%)': 'mean',
    'windspeed_10m (km/h)': 'mean'
}).reset_index()

weather_daily.columns = [
    'date',
    'temp',
    'precipitation',
    'rain',
    'cloudcover',
    'windspeed'
]

weather_daily = weather_daily.ffill()

In [52]:
# 4. Merge weather data
df['date'] = df['started_at'].dt.date

df = df.merge(weather_daily, on='date', how='left')

df.head()

print(df.columns)


Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual', 'trip_duration_min', 'hour', 'day_of_week', 'month',
       'is_weekend', 'season', 'date', 'temp', 'precipitation', 'rain',
       'cloudcover', 'windspeed'],
      dtype='str')


### Export to .csv for analysis and Gradio

In [53]:
df.head()
df.to_csv('data/processed_data.csv', index=False)